# Dynamic Entité Incident Table
Upload a JSON file of incidents. Use the dropdown to select an entity and view their incident repartition by category.

In [20]:
import pandas as pd
import json
import re
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, HTML

!pip install ipywidgets -q

def load_json_to_df(path):
    for encoding in ('utf-8', 'latin-1'):
        try:
            with open(path, encoding=encoding) as f:
                data = json.load(f)
            return pd.json_normalize(data)
        except Exception:
            continue
    raise ValueError(f'Unable to load JSON file: {path}')

def normalize_columns(df):
    mapping = {}
    for col in df.columns:
        normalized = re.sub(r'[^a-z0-9]', '', str(col).lower())
        if normalized in ('client', 'clientname', 'tenant', 'societe', 'company', 'entite', 'entité'):
            mapping[col] = 'Entité'
        elif normalized in ('serveur', 'server', 'host', 'hostname', 'apparail'):
            mapping[col] = 'Serveur'
        elif normalized in ('titre', 'title', 'subject', 'objet', 'description', 'summary'):
            mapping[col] = 'Titre'
    df = df.rename(columns=mapping)
    for target in ('Entité', 'Serveur', 'Titre'):
        if target not in df.columns:
            df[target] = pd.NA
    return df

def categorize_row(row):
    titre = str(row.get('Titre', '') or '')
    serveur = str(row.get('Serveur', '') or '')
    combined = f'{titre} {serveur}'
    if re.search(r'ntnx|nutanix', combined, flags=re.I):
        return 'Nutanix'
    if re.search(r'cpu', titre, flags=re.I):
        return 'CPU Issue'
    if re.search(r'memory|ram', titre, flags=re.I):
        return 'Memory Issue'
    if re.search(r'disk|storage', titre, flags=re.I):
        return 'Disk'
    if re.search(r'network|nic|link', titre, flags=re.I):
        return 'Network'
    return 'OS / Autres'


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: c:\Users\aelazmi2\Desktop\glpi_pipeline.py\.venv\Scripts\python.exe -m pip install --upgrade pip


In [21]:
# Enter the path to your JSON file and run this cell
import os

json_path = "Synthèse DC - Incident.json"  # Change this to your JSON file path

if os.path.exists(json_path):
    df = load_json_to_df(json_path)
    df = normalize_columns(df)
    df['Categorie'] = df.apply(categorize_row, axis=1)
    entities = sorted(df['Entité'].dropna().unique())
    print(f"✅ Loaded {len(df)} incidents")
    print(f"📋 Found {len(entities)} entités")
    entity_dropdown = widgets.Dropdown(description='Entité:', options=entities)
    entity_dropdown.df = df
    entity_dropdown.value = entities[0] if entities else None
    entity_dropdown.value = entities[0] if entities else None
    entity_dropdown.df = df
else:
    print(f"❌ File not found: {json_path}")
    print("Please update the json_path variable above with your JSON file path")

✅ Loaded 29122 incidents
📋 Found 10 entités


In [22]:
# Dropdown and dynamic table
output = widgets.Output()

def update_table(change):
    output.clear_output()
    df = getattr(entity_dropdown, 'df', None)
    if df is None or not entity_dropdown.value:
        return
    filtered = df[df['Entité'] == entity_dropdown.value]
    category_order = [
        'CPU Issue', 'Memory Issue', 'Disk', 'Network', 'Nutanix', 'OS / Autres'
    ]
    display_map = {
        'CPU Issue': 'CPU Issue',
        'Memory Issue': 'Memory Issue',
        'OS / Autres': 'OS service',
        'Nutanix': 'Nutanix Issue',
        'Disk': 'VM availability / Disk / MSSQL / autres',
        'Network': 'Network',
    }
    counts = filtered['Categorie'].value_counts().reindex(category_order, fill_value=0)
    total = counts.sum()
    # Convert Series to DataFrame explicitly to avoid KeyError
    counts_df = counts.reset_index()
    counts_df.columns = ['Categorie', 'Volume']
    table_df = (
        counts_df.assign(Categorie=lambda df: df['Categorie'].map(display_map).fillna(df['Categorie']))
        .assign(Part_du_total=lambda df: (df['Volume'] / total * 100).round(0).astype(int).astype(str) + ' %')
        .loc[lambda df: df['Categorie'].isin([
            'CPU Issue', 'Memory Issue', 'OS service', 'Nutanix Issue', 'VM availability / Disk / MSSQL / autres'
        ])]
        .loc[:, ['Categorie', 'Volume', 'Part_du_total']]
    )
    with output:
        display(HTML(table_df.to_html(index=False, escape=False)))

entity_dropdown.observe(update_table, names='value')
display(entity_dropdown, output)

# Initial display
update_table(None)

Dropdown(description='Entité:', options=('Data_Center > Avocat', 'Data_Center > CG Park', 'Data_Center > CMS',…

Output()

In [23]:
# Export to PowerPoint with interactive dropdown
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.dml.color import RGBColor
import os

def create_interactive_ppt(df, output_path="incident_analysis.pptx"):
    """Create a PowerPoint with all entity data and a descriptive textbox per slide."""
    prs = Presentation()
    prs.slide_width = Inches(13.333)
    prs.slide_height = Inches(7.5)

    entities = sorted(df['Entité'].dropna().unique())

    category_order = ['CPU Issue','Memory Issue','Disk','Network','Nutanix','OS / Autres']
    display_map = {
        'CPU Issue': 'CPU Issue',
        'Memory Issue': 'Memory Issue',
        'OS / Autres': 'OS service',
        'Nutanix': 'Nutanix Issue',
        'Disk': 'VM availability / Disk / MSSQL / autres',
        'Network': 'Network',
    }

    # Title slide
    title_slide_layout = prs.slide_layouts[0]
    slide = prs.slides.add_slide(title_slide_layout)
    slide.shapes.title.text = "Incident Analysis by Entity"
    subtitle = slide.placeholders[1]
    subtitle.text = f"Total: {len(df)} incidents across {len(entities)} entities"

    for entity in entities:
        slide = prs.slides.add_slide(prs.slide_layouts[6])
        title_box = slide.shapes.add_textbox(Inches(0.5), Inches(0.3), Inches(12), Inches(0.8))
        title_frame = title_box.text_frame
        title_frame.text = entity
        title_para = title_frame.paragraphs[0]
        title_para.font.size = Pt(24)
        title_para.font.bold = True
        title_para.font.color.rgb = RGBColor(0,51,102)

        filtered = df[df['Entité'] == entity]
        counts = filtered['Categorie'].value_counts().reindex(category_order, fill_value=0)
        total = int(counts.sum())

        # Build table data using display names
        table_data = [['Catégorie','Volume','Part du total']]
        for key in category_order:
            display_cat = display_map.get(key, key)
            vol = int(counts.get(key,0))
            pct = f"{int(round((vol/total)*100,0))}%" if total > 0 else "0%"
            table_data.append([display_cat, str(vol), pct])

        rows = len(table_data)
        cols = len(table_data[0])
        left, top, width, height = Inches(1), Inches(1.5), Inches(11), Inches(0.8)
        table = slide.shapes.add_table(rows, cols, left, top, width, height).table
        table.columns[0].width = Inches(5)
        table.columns[1].width = Inches(2)
        table.columns[2].width = Inches(2)

        for i, row_data in enumerate(table_data):
            for j, cell_text in enumerate(row_data):
                cell = table.cell(i,j)
                cell.text = cell_text
                if i == 0:
                    cell.fill.solid()
                    cell.fill.fore_color.rgb = RGBColor(0,51,102)
                    para = cell.text_frame.paragraphs[0]
                    para.font.color.rgb = RGBColor(255,255,255)
                    para.font.bold = True
                else:
                    if i % 2 == 0:
                        cell.fill.solid()
                        cell.fill.fore_color.rgb = RGBColor(240,240,240)

        # Description summarizing top categories (use display names)
        nonzero = counts[counts > 0]
        if total > 0 and not nonzero.empty:
            top_cats = nonzero.sort_values(ascending=False).head(2)
            parts = []
            for k, v in top_cats.items():
                display_cat = display_map.get(k,k)
                pct = int(round(v / total * 100, 0))
                parts.append(f"{display_cat}: {v} ({pct}% )")
            desc = f"Total incidents for {entity}: {total}. Top categories: {', '.join(parts)}."
        else:
            desc = f"No incidents recorded for {entity}."

        desc_box = slide.shapes.add_textbox(Inches(1), Inches(4.0), Inches(11), Inches(0.6))
        desc_frame = desc_box.text_frame
        desc_frame.text = desc
        desc_para = desc_frame.paragraphs[0]
        desc_para.font.size = Pt(12)
        desc_para.font.color.rgb = RGBColor(51,51,51)

        total_box = slide.shapes.add_textbox(Inches(1), Inches(5.5), Inches(11), Inches(0.5))
        total_frame = total_box.text_frame
        total_frame.text = f"Total incidents for {entity}: {total}"
        total_para = total_frame.paragraphs[0]
        total_para.font.size = Pt(14)
        total_para.font.italic = True

    # Instructions slide
    instr = prs.slides.add_slide(prs.slide_layouts[1])
    instr.shapes.title.text = "How to Navigate"
    body = instr.placeholders[1]
    body.text = ("Use PowerPoint's slide navigation to browse through each entity's data.\n\n"
                 "Each slide shows the incident breakdown for one entity.\n\n"
                 f"Total entities: {len(entities)}\n"
                 f"Total incidents: {len(df)}")

    os.makedirs('analyse_output', exist_ok=True)
    outpath = os.path.join('analyse_output', output_path)
    prs.save(outpath)
    print(f"✅ PowerPoint saved: {outpath}")
    print(f"📊 Contains {len(entities)} entity slides + title + instructions")
    return outpath

if 'df' in globals():
    create_interactive_ppt(df, 'incident_analysis.pptx')
else:
    print("⚠️ Please run the data-loading cell first to create `df`.")


✅ PowerPoint saved: analyse_output\incident_analysis.pptx
📊 Contains 10 entity slides + title + instructions
